# データ探索

## 目的
山本由伸2025年基準で順位など成績を見る

## 粒度

- 1球：球種ごとの球速・変化量・回転量・コースなどのストライク・ボール・アウト率など
- 試合：シーズン成績

## 期間
- レギュラー
- ポスト
- WS



# 共通セットアップ

In [ ]:
import importlib
import json

import polars as pl

import analysis_project.statcast_columns as statcast_columns
from analysis_project.paths import data_dir

# src 更新後も Jupyter が古い statcast_columns を握ることがある
importlib.reload(statcast_columns)
rename_arsenal_report_columns = statcast_columns.rename_arsenal_report_columns
rename_statcast_columns = statcast_columns.rename_statcast_columns
statcast_column_rename_summary = statcast_columns.statcast_column_rename_summary


YAMAMOTO_MLBAM = 808967


# Statcast

In [ ]:
# データの読み込み
statcast_regular_path = data_dir() / "external" / "statcast" / "2025_regular.parquet"
statcast_post_path = data_dir() / "external" / "statcast" / "2025_post.parquet"


# dfの作成
df_statcast_regular = pl.read_parquet(statcast_regular_path)
df_statcast_post = pl.read_parquet(statcast_post_path)

# dfの結合
df_statcast = pl.concat([df_statcast_regular, df_statcast_post], how="vertical")

# 山本由伸のデータ抽出
df_statcast_ja = rename_statcast_columns(df_statcast)

df_regular_yamamoto = df_statcast.filter(pl.col("pitcher") == YAMAMOTO_MLBAM)
df_post_yamamoto = df_statcast.filter(pl.col("pitcher") == YAMAMOTO_MLBAM)
df_all_yamamoto = df_statcast.filter(pl.col("pitcher") == YAMAMOTO_MLBAM)
df_all_yamamoto_ja = rename_statcast_columns(df_all_yamamoto)


# 列名早見表
# Statcast DataFrame 列名早見表

1 行 = 1 球（pybaseball / `2025_*`.parquet）。列名は英語のまま分析する。

## 試合・状況

| 列名 | 早見（日本語） |
|------|----------------|
| `game_date` | 試合日 |
| `game_year` | 試合年 |
| `game_pk` | 試合 ID |
| `game_type` | 試合種別（R=レギュラー, F/D/L/W=ポスト等） |
| `home_team` | ホーム球団略称 |
| `away_team` | アウェイ球団略称 |
| `inning` | ピッチ前イニング |
| `inning_topbot` | 表 Top / 裏 Bot |
| `outs_when_up` | ピッチ前アウト数 |
| `at_bat_number` | 試合内打席通算番号 |
| `pitch_number` | 打席内の球数 |
| `balls` | ピッチ前ボール数 |
| `strikes` | ピッチ前ストライク数 |

## 選手・ID

| 列名 | 早見（日本語） |
|------|----------------|
| `player_name` | 選手名（表示用） |
| `pitcher` | 投手 MLBAM ID |
| `batter` | 打者 MLBAM ID |
| `p_throws` | 投手腕 R/L |
| `stand` | 打席側 R/L |
| `on_1b` / `on_2b` / `on_3b` | ピッチ前走者 ID（塁ごと） |
| `fielder_2`〜`fielder_9` | ピッチ前守備者 ID（2=捕手…9=右翼） |
| `age_pit` / `age_bat` | 年齢（12/31 基準） |
| `age_pit_legacy` / `age_bat_legacy` | 年齢（6/30 基準・旧） |
| `n_thruorder_pitcher` | 投手が打順何周目か |
| `n_priorpa_thisgame_player_at_bat` | 打者の当試合 prior 打席数 |
| `pitcher_days_since_prev_game` | 投手・前試合からの日数 |
| `batter_days_since_prev_game` | 打者・前試合からの日数 |
| `pitcher_days_until_next_game` | 投手・次試合までの日数 |
| `batter_days_until_next_game` | 打者・次試合までの日数 |

## 球種・投球

| 列名 | 早見（日本語） |
|------|----------------|
| `pitch_type` | 球種コード（FF, SL…） |
| `pitch_name` | 球種英語名 |
| `release_speed` | 球速 mph |
| `effective_speed` | 延伸込み換算球速 mph |
| `release_spin_rate` | 回転 rpm |
| `spin_axis` | 回転軸（度） |
| `release_extension` | リリース延伸 ft |
| `release_pos_x` / `y` / `z` | リリース位置 ft |
| `pfx_x` / `pfx_z` | 球の動き（水平/垂直）ft |
| `vx0`/`vy0`/`vz0` | 50ft 付近初速 ft/s |
| `ax`/`ay`/`az` | 50ft 付近加速度 ft/s² |
| `api_break_z_with_gravity` | 重力込み垂直ブレーク |
| `api_break_x_arm` | 腕側ブレーク in |
| `api_break_x_batter_in` | 打者内側ブレーク in |
| `arm_angle` | 腕角度 |

## コース

| 列名 | 早見（日本語） |
|------|----------------|
| `plate_x` / `plate_z` | 本塁通過位置 |
| `zone` | ストライクゾーン番号 |
| `sz_top` / `sz_bot` | 打者ゾーン上下 |

## 結果

| 列名 | 早見（日本語） |
|------|----------------|
| `type` | B/S/X（ボール/ストライク/インプレイ） |
| `description` | その球の結果 |
| `events` | 打席終了イベント |
| `des` | 打席説明文（GameDay） |
| `sv_id` | プレーイベント ID |

## 打球

| 列名 | 早見（日本語） |
|------|----------------|
| `hit_location` | 最初に触れた守備位置 |
| `bb_type` | ゴロ/ライナー/フライ等 |
| `hc_x` / `hc_y` | 打球座標 |
| `hit_distance_sc` | 推定飛距離 ft |
| `launch_speed` / `launch_angle` | 初速 mph / 角度 ° |
| `launch_speed_angle` | 初速×角度ゾーン（Barrel 等） |
| `hyper_speed` | Adjusted EV |
| `estimated_ba_using_speedangle` | 推定打率 |
| `estimated_woba_using_speedangle` | 推定 wOBA |
| `estimated_slg_using_speedangle` | 推定長打率 |

## 価値・期待値

| 列名 | 早見（日本語） |
|------|----------------|
| `woba_value` / `woba_denom` | wOBA 値・分母 |
| `babip_value` | BABIP 用値 |
| `iso_value` | ISO 用値 |
| `delta_run_exp` | 得点期待値の変化 |
| `delta_pitcher_run_exp` | 投手視点 RE 変化 |
| `delta_home_win_exp` | ホーム勝率期待の変化 |
| `home_win_exp` / `bat_win_exp` | ピッチ前勝率期待 |

## スコア

| 列名 | 早見（日本語） |
|------|----------------|
| `home_score` / `away_score` | ピッチ前得点 |
| `bat_score` / `fld_score` | ピッチ前攻撃/守備得点 |
| `post_home_score` / `post_away_score` | ピッチ後 |
| `post_bat_score` / `post_fld_score` | ピッチ後攻撃/守備 |
| `home_score_diff` / `bat_score_diff` | 得点差 |

## 守備・バットトラッキング

| 列名 | 早見（日本語） |
|------|----------------|
| `if_fielding_alignment` | 内野シフト |
| `of_fielding_alignment` | 外野シフト |
| `bat_speed` | バット速度 mph |
| `swing_length` | スイング長 ft |
| `miss_distance` | ミス距離 in |
| `attack_angle` / `attack_direction` / `swing_path_tilt` | スイング・接触角度 |
| `intercept_ball_minus_batter_pos_x_inches` | 接触点−打者重心 X in |
| `intercept_ball_minus_batter_pos_y_inches` | 接触点−打者重心 Y in |

## 非推奨（分析では通常使わない）

| 列名 | 早見（日本語） |
|------|----------------|
| `spin_dir` | 旧トラッキング |
| `spin_rate_deprecated` | 旧回転 → `release_spin_rate` |
| `break_angle_deprecated` / `break_length_deprecated` | 旧ブレーク |
| `tfs_deprecated` / `tfs_zulu_deprecated` | 旧タイムスタンプ |
| `umpire` | 旧審判 ID |

In [ ]:
PITCHER_MLBAM = 808967  # 山本由伸。全投手なら後述の group_by を追加
EXCLUDE_PITCH_TYPES = ("UN", "PO")

# Whiff / CSW / スイング（description 固定 — レポート脚注に書く）
WHIFF_DESCRIPTIONS = ("swinging_strike", "swinging_strike_blocked", "missed_bunt")
CALLED_STRIKE_DESCRIPTIONS = ("called_strike", "automatic_strike")
SWING_DESCRIPTIONS = (
    "swinging_strike",
    "swinging_strike_blocked",
    "foul",
    "foul_tip",
    "foul_bunt",
    "bunt_foul_tip",
    "hit_into_play",
    "missed_bunt",
)
# 打席終了で打者アウト（events 固定）
OUT_EVENTS = (
    "strikeout",
    "field_out",
    "force_out",
    "grounded_into_double_play",
    "double_play",
    "triple_play",
    "strikeout_double_play",
    "sac_fly",
    "sac_bunt",
    "fielders_choice_out",
    "sac_fly_double_play",
)

In [ ]:
cols = [
    "pitcher",
    "pitch_type",
    "game_pk",
    "at_bat_number",
    "pitch_number",
    "description",
    "type",
    "zone",
    "events",
    "release_speed",
    "effective_speed",
    "release_spin_rate",
    "pfx_x",
    "pfx_z",
    "release_pos_x",
    "release_pos_z",
    "release_extension",
    "spin_axis",
    "arm_angle",
    "launch_speed",
    "estimated_woba_using_speedangle",
    "woba_value",
    "woba_denom",
]

statcast_path = data_dir() / "external" / "statcast" / "2025_regular.parquet"
df = pl.read_parquet(statcast_path, columns=cols)

# 集計は英語列名の df。表示用に df_ja（このセル末尾で作成）
df = df.filter(
    (pl.col("pitcher") == PITCHER_MLBAM)
    & pl.col("pitch_type").is_not_null()
    & ~pl.col("pitch_type").is_in(EXCLUDE_PITCH_TYPES)
)

df = df.with_columns(
    pl.col("zone").is_between(1, 9).alias("in_zone"),
    pl.col("description").is_in(WHIFF_DESCRIPTIONS).alias("is_whiff"),
    pl.col("description").is_in(CALLED_STRIKE_DESCRIPTIONS).alias("is_called_strike"),
    pl.col("description").is_in(SWING_DESCRIPTIONS).alias("is_swing"),
    (pl.col("type") == "S").alias("is_strike_type"),
    (pl.col("description") == "hit_into_play").alias("is_in_play"),
    pl.col("events").is_in(OUT_EVENTS).alias("is_batter_out"),
)

# 打席の最終球（球種別打席終了アウト率用）
df = df.with_columns(
    pl.col("pitch_number")
    .max()
    .over("game_pk", "at_bat_number")
    .alias("last_pitch_in_pa")
)
df = df.with_columns(
    (pl.col("pitch_number") == pl.col("last_pitch_in_pa")).alias("is_pa_terminal_pitch")
)

# ゾーン外スイング（Chase%）
df = df.with_columns(
    (~pl.col("in_zone") & pl.col("is_swing")).alias("is_chase_swing"),
    (~pl.col("in_zone")).alias("is_outside_zone"),
)

df_ja = rename_statcast_columns(df)
print(statcast_column_rename_summary(df))

In [ ]:
n_pitcher = df.height

report = (
    df.group_by("pitch_type")
    .agg(
        # 配球
        pl.len().alias("n_pitches"),
        (pl.len() / n_pitcher).alias("usage"),
        # 物理（平均・中央値）
        pl.col("release_speed").mean().alias("speed_mean"),
        pl.col("release_speed").median().alias("speed_median"),
        pl.col("release_spin_rate").mean().alias("spin_mean"),
        pl.col("release_spin_rate").median().alias("spin_median"),
        pl.col("pfx_x").mean().alias("pfx_x_mean"),
        pl.col("pfx_x").median().alias("pfx_x_median"),
        pl.col("pfx_z").mean().alias("pfx_z_mean"),
        pl.col("pfx_z").median().alias("pfx_z_median"),
        pl.col("release_extension").mean().alias("extension_mean"),
        pl.col("spin_axis").mean().alias("spin_axis_mean"),
        pl.col("release_pos_x").mean().alias("release_pos_x_mean"),
        pl.col("release_pos_z").mean().alias("release_pos_z_mean"),
        # 安定（SD）
        pl.col("release_pos_x").std().alias("release_pos_x_sd"),
        pl.col("release_pos_z").std().alias("release_pos_z_sd"),
        pl.col("release_speed").std().alias("speed_sd"),
        # プロセス（率）
        pl.col("in_zone").mean().alias("zone_pct"),
        pl.col("is_strike_type").mean().alias("strike_pct"),  # type==S の割合
        (
            (pl.col("is_called_strike") | pl.col("is_whiff")).sum() / pl.len()
        ).alias("csw_pct"),
        (pl.col("is_whiff").sum() / pl.col("is_swing").sum()).alias("whiff_pct"),
        (pl.col("is_swing").mean()).alias("swing_pct"),
        (
            pl.col("is_chase_swing").sum() / pl.col("is_outside_zone").sum()
        ).alias("chase_pct"),
        # インプレイの打たれ質
        pl.col("estimated_woba_using_speedangle")
        .filter(pl.col("is_in_play"))
        .mean()
        .alias("xwoba_mean_in_play"),
        pl.col("launch_speed")
        .filter(pl.col("is_in_play"))
        .mean()
        .alias("ev_mean_in_play"),
        # 打席終了（最終球のみ）
        pl.col("is_pa_terminal_pitch").sum().alias("n_pa_terminal"),
        (
            pl.col("is_batter_out").filter(pl.col("is_pa_terminal_pitch")).sum()
            / pl.col("is_pa_terminal_pitch").sum()
        ).alias("pa_terminal_out_pct"),
        (
            pl.col("woba_value")
            .filter(pl.col("is_pa_terminal_pitch") & (pl.col("woba_denom") > 0))
            .mean()
        ).alias("woba_mean_pa_terminal"),
    )
    .sort("n_pitches", descending=True)
)

# False: pitch_type + pitch_name（英語）のみ / True: 先頭に「球種」（日本語）
USE_JA_PITCH_LABEL = False

pitch_map = pl.read_csv(
    data_dir() / "interim" / "mappings" / "statcast_pitch_type_ja.csv"
)
report_ja = report.join(
    pitch_map.select(
        "pitch_type",
        pl.col("pitch_name_en").alias("pitch_name"),
        "pitch_name_ja",
    ),
    on="pitch_type",
    how="left",
)

if USE_JA_PITCH_LABEL:
    report_ja = report_ja.rename({"pitch_name_ja": "球種"})
    id_cols = ["球種", "pitch_type", "pitch_name"]
else:
    id_cols = ["pitch_type", "pitch_name"]

metric_cols = [
    c for c in report_ja.columns if c not in id_cols and c != "pitch_name_ja"
]
report_ja = report_ja.select(id_cols + metric_cols)

# 集計列（n_pitches 等）は statcast_column_ja には無い → 専用マッピングを使う
report_ja = rename_arsenal_report_columns(report_ja)

report_ja

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "MS Gothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# ストライクゾーン横半幅（本塁板17インチの半分・ft）
STRIKE_ZONE_HALF_WIDTH_FT = (17 / 12) / 2

plot_df = df_all_yamamoto.filter(
    pl.col("plate_x").is_not_null()
    & pl.col("plate_z").is_not_null()
    & pl.col("stand").is_in(["L", "R"])
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
stand_titles = {"R": "右打者 (stand=R)", "L": "左打者 (stand=L)"}

for ax, stand in zip(axes, ["R", "L"], strict=True):
    sub = plot_df.filter(pl.col("stand") == stand)
    x = sub["plate_x"].to_numpy()
    z = sub["plate_z"].to_numpy()
    hb = ax.hexbin(
        x,
        z,
        gridsize=28,
        cmap="Blues",
        mincnt=1,
        linewidths=0.2,
        edgecolors="face",
    )
bounds = sub.select(
    pl.col("sz_bot").median().cast(pl.Float64).alias("sz_bot"),
    pl.col("sz_top").median().cast(pl.Float64).alias("sz_top"),
).row(0, named=True)
sz_bot = bounds["sz_bot"]
sz_top = bounds["sz_top"]
if sz_bot is not None and sz_top is not None:
    zone = Rectangle(
        (-STRIKE_ZONE_HALF_WIDTH_FT, float(sz_bot)),
        2 * STRIKE_ZONE_HALF_WIDTH_FT,
        float(sz_top) - float(sz_bot),
            linewidth=1.5,
            edgecolor="crimson",
            facecolor="none",
            linestyle="--",
        )
    ax.add_patch(zone)
    ax.set_title(f"{stand_titles[stand]}  n={sub.height:,}")
    ax.set_xlabel("plate_x（捕手視点・ft）")
    ax.set_ylabel("plate_z（捕手視点・ft）")
    ax.set_aspect("equal")

fig.colorbar(hb, ax=axes, label="投球数（ビン内）", shrink=0.85)
fig.suptitle("山本由伸：本塁通過位置の密度", y=1.02)
fig.tight_layout()